# Hybrid AI Shorts Generator — Google Colab (Cloud GPU)

Poora project Colab ke **free T4 GPU** par chalega — Whisper transcription aur Laya dono GPU par, isliye WSL ke CPU run se kai guna fast.

**Kaise chalana hai:** upar menu se **Runtime → Run all** dabao. Pehli baar setup me ~8–12 minute lagenge (packages + models download). Uske baad neeche last cell me app ka link milega — wahi tumhara Shorts Generator hai.

Dhyaan do:
- Runtime type **GPU (T4)** hona chahiye: Runtime → Change runtime type → T4 GPU.
- Colab session band hote hi sab delete ho jata hai — Shorts ban jayein to **Export page se ZIP download** kar lena. (Projects bachane hon to neeche wali Drive wali cell chala lena.)
- Session zyada se zyada ~12 ghante chalta hai (free tier me pehle bhi band ho sakta hai).

In [ ]:
# Step 1 — GPU check (T4 dikhna chahiye)
!nvidia-smi
import torch
print('Torch CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — Runtime type GPU par badlo')

In [ ]:
# Step 2 (OPTIONAL) — Google Drive mount: projects/renders Drive me save honge,
# session band hone par bhi nahi udenge. Zarurat na ho to ye cell skip kar do.
# Neeche wali dono lines ke aage se # hatao aur cell chalao:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Step 3 — Code clone karo
import os
if not os.path.exists('/content/hybrid-shorts-generator'):
    !git clone https://github.com/Khelendrameena/hybrid-shorts-generator.git /content/hybrid-shorts-generator
else:
    !cd /content/hybrid-shorts-generator && git pull
%cd /content/hybrid-shorts-generator

In [ ]:
# Step 4 — Python packages install (pehli baar ~5-8 min). Colab me ffmpeg pehle se hota hai.
!pip install -q -r requirements.txt
!ffmpeg -version | head -1

In [ ]:
# Step 5 — Ollama (titles/hooks ke liye local LLM) install + model pull
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /content/ollama.log 2>&1 &
import time, urllib.request
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(2)
!ollama pull qwen3:4b

In [ ]:
# Step 6 — Backend start (GPU). Agar Step 2 me Drive mount kiya tha to
# DATA_DIR Drive par jayega, warna data session ke saath delete hoga.
import os
os.environ['USE_TF'] = '0'
if os.path.exists('/content/drive/MyDrive'):
    os.environ['DATA_DIR'] = '/content/drive/MyDrive/hybrid-shorts-data'
    print('DATA_DIR =', os.environ['DATA_DIR'], '(Drive — projects safe rahenge)')
!nohup python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000 > /content/backend.log 2>&1 &
import time, urllib.request, json
for i in range(60):
    try:
        h = json.load(urllib.request.urlopen('http://localhost:8000/api/health', timeout=3))
        print('Backend UP:', h['ok'], '| Ollama:', h['ollama'])
        break
    except Exception:
        time.sleep(3)
else:
    print('Backend start nahi hua — neeche wali cell se log dekho')
!tail -3 /content/backend.log

In [ ]:
# Step 7 — Backend ke liye public tunnel (cloudflare) + frontend ko uska address dena
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!nohup cloudflared tunnel --url http://localhost:8000 > /content/cf8000.log 2>&1 &
import re, time
api_url = None
for i in range(30):
    time.sleep(2)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf8000.log').read())
    if m:
        api_url = m[0]; break
print('API URL:', api_url)
open('/content/hybrid-shorts-generator/frontend/.env.local', 'w').write(f'NEXT_PUBLIC_API_URL={api_url}\n')
print('frontend/.env.local set ho gaya')

In [ ]:
# Step 8 — Frontend build (API URL ke saath) — ~2-3 min
!cd /content/hybrid-shorts-generator/frontend && npm install --silent && npm run build

In [ ]:
# Step 9 — Frontend start + uska tunnel = TUMHARA APP LINK
!cd /content/hybrid-shorts-generator/frontend && nohup npm run start > /content/frontend.log 2>&1 &
!nohup cloudflared tunnel --url http://localhost:3000 > /content/cf3000.log 2>&1 &
import re, time
app_url = None
for i in range(30):
    time.sleep(2)
    try:
        m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf3000.log').read())
    except FileNotFoundError:
        continue
    if m:
        app_url = m[0]; break
print('\n==============================================')
print('APP LINK (browser me kholo):', app_url)
print('==============================================')
print('Note: ye link sirf tab tak chalega jab tak ye Colab session chal raha hai.')

## Ho gaya! Ab kya karna hai

1. Upar wale **APP LINK** ko browser me kholo — wahi tumhara Shorts Generator hai.
2. YouTube URL daalo → Process karo. Captions wali video par transcript seconds me aayega; bina captions wali par Whisper **T4 GPU** par chalega (CPU se kai guna fast).
3. Clips Approve karo → Render karo → **Export page se ZIP download** kar lo (session band = sab delete).

Agar kuch atke: backend ka log dekhne ke liye kisi cell me `!tail -30 /content/backend.log` chalao aur screenshot bhej do.